In [2]:
import pandas as pd

df = pd.read_csv("한파일수.csv", encoding="cp949", header=None, names=range(15), skip_blank_lines=False)

print("데이터 크기:", df.shape)
display(df.iloc[10:20, :4])

cold = df.iloc[16:, :3].copy()
cold.columns = ["연도", "날짜", "지점명"]

print("\n관측값 표 크기:", cold.shape)
display(cold.head())
display(cold.tail())

데이터 크기: (1697, 15)


,0,1,2,3
10,2025,1.5,2.4,0.1
11,평균,2.9,1.0,0.0
12,NaN,NaN,NaN,NaN
13,NaN,NaN,NaN,NaN
14,한파일수 관측값,NaN,NaN,NaN
15,연도,날짜,지점,NaN
16,2021,0101,의성,NaN
17,2021,0101,봉화,NaN
18,2021,0101,양평,NaN
19,2021,0101,이천,NaN



관측값 표 크기: (1681, 3)


,연도,날짜,지점명
16,2021,0101,의성
17,2021,0101,봉화
18,2021,0101,양평
19,2021,0101,이천
20,2021,0101,제천


,연도,날짜,지점명
1692,2025,1227,홍천
1693,2025,1227,대관령
1694,2025,1227,춘천
1695,2025,1227,철원
1696,2025,1231,대관령


In [3]:
cold["일시"] = pd.to_datetime(cold["연도"] + cold["날짜"], format="%Y%m%d")
cold_count = cold.groupby(["지점명", "연도"]).size().reset_index(name="목록_한파일수")

temp = pd.read_csv("기상_일별_최고최저기온_정제.csv", encoding="utf-8-sig")
temp["일시"] = pd.to_datetime(temp["일시"])
temp["연도"] = temp["일시"].dt.year.astype(str)

cold_temp = temp[temp["최저기온(°C)"] <= -12].groupby(["지점명", "연도"]).size().reset_index(name="기온파일_한파일수")

check = cold_count.merge(cold_temp, on=["지점명", "연도"], how="outer").fillna(0)

print("비교 행 수:", len(check))
print("불일치 행 수:", (check["목록_한파일수"] != check["기온파일_한파일수"]).sum())
print("목록이 더 큰 행:", (check["목록_한파일수"] > check["기온파일_한파일수"]).sum())
print("총합 - 목록:", int(check["목록_한파일수"].sum()), "/ 기온파일:", int(check["기온파일_한파일수"].sum()))

비교 행 수: 297
불일치 행 수: 188
목록이 더 큰 행: 0
총합 - 목록: 1681 / 기온파일: 2575


In [4]:
obs = temp.groupby(["지점", "지점명", "시도", "연도"]).size().reset_index(name="관측일수")

cold_final = obs.merge(cold_temp, on=["지점명", "연도"], how="left")
cold_final["기온파일_한파일수"] = cold_final["기온파일_한파일수"].fillna(0).astype(int)
cold_final = cold_final.rename(columns={"기온파일_한파일수": "한파일수"})

print("크기:", cold_final.shape)
print("지점 개수:", cold_final["지점명"].nunique())
print("한파일수 0인 행:", (cold_final["한파일수"] == 0).sum())
print("한파일수 합계:", cold_final["한파일수"].sum())
display(cold_final.sort_values("한파일수", ascending=False).head(3))

크기: (481, 6)
지점 개수: 97
한파일수 0인 행: 184
한파일수 합계: 2575


,지점,지점명,시도,연도,관측일수,한파일수
26,100,대관령,강원특별자치도,2022,365,58
11,95,철원,강원특별자치도,2022,365,53
21,99,파주,경기도,2022,365,47


In [5]:
winter = temp[temp["일시"].dt.month.isin([11, 12, 1, 2, 3])]
winter_obs = winter.dropna(subset=["최저기온(°C)"]).groupby(["지점명", "연도"]).size().reset_index(name="겨울관측일수")

cold_final = cold_final.merge(winter_obs, on=["지점명", "연도"], how="left")
cold_final["겨울관측일수"] = cold_final["겨울관측일수"].fillna(0).astype(int)

cold_final["겨울일수"] = cold_final["연도"].apply(lambda y: 152 if int(y) % 4 == 0 else 151)
cold_final["겨울부족일수"] = cold_final["겨울일수"] - cold_final["겨울관측일수"]

short_w = cold_final[cold_final["겨울부족일수"] > 0]

print("겨울철 관측이 부족한 행:", len(short_w))
print("그중 5일 이상 부족:", (short_w["겨울부족일수"] >= 5).sum())
display(short_w.sort_values("겨울부족일수", ascending=False).head(10)[["지점명", "연도", "한파일수", "겨울관측일수", "겨울부족일수"]])

겨울철 관측이 부족한 행: 18
그중 5일 이상 부족: 2


,지점명,연도,한파일수,겨울관측일수,겨울부족일수
205,서청주,2023,4,61,90
478,북부산,2023,1,141,10
385,의령군,2023,4,147,4
29,대관령,2025,38,147,4
257,홍천,2025,16,147,4
188,완도,2024,0,148,4
267,정선군,2025,21,148,3
399,진도군,2022,0,148,3
479,북부산,2024,0,150,2
167,통영,2023,0,149,2


In [6]:
cold_final["겨울관측_주의"] = cold_final["겨울부족일수"] >= 5

print("주의 표시된 행:", cold_final["겨울관측_주의"].sum())
display(cold_final[cold_final["겨울관측_주의"]][["지점명", "연도", "한파일수", "겨울관측일수", "겨울부족일수"]])

주의 표시된 행: 2


,지점명,연도,한파일수,겨울관측일수,겨울부족일수
205,서청주,2023,4,61,90
478,북부산,2023,1,141,10


In [7]:
final_cols = ["지점", "지점명", "시도", "연도", "한파일수", "관측일수", "겨울관측일수", "겨울부족일수", "겨울관측_주의"]
cold_out = cold_final[final_cols].sort_values(["지점", "연도"]).reset_index(drop=True)

print("크기:", cold_out.shape)
print("지점 개수:", cold_out["지점명"].nunique())
print("(지점, 연도) 중복:", cold_out.duplicated(["지점", "연도"]).sum())
print("결측치 개수:\n", cold_out.isna().sum())
print("한파일수 합계:", cold_out["한파일수"].sum())
print("한파일수 최소/최대:", cold_out["한파일수"].min(), cold_out["한파일수"].max())

cold_out.to_csv("한파일수_정제.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료")

크기: (481, 9)
지점 개수: 97
(지점, 연도) 중복: 0
결측치 개수:
 지점         0
지점명        0
시도         0
연도         0
한파일수       0
관측일수       0
겨울관측일수     0
겨울부족일수     0
겨울관측_주의    0
dtype: int64
한파일수 합계: 2575
한파일수 최소/최대: 0 58

저장 완료
